In [1]:
import json
import random
from pathlib import Path
import numpy as np
import open3d as o3d


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def collect_nodes_by_name(nodes, target_name, results):
    for node in nodes:
        if node.get("name") == target_name:
            results.append(node)
        if "children" in node:
            collect_nodes_by_name(node["children"], target_name, results)


def load_mesh_o3d(path):
    mesh = o3d.io.read_triangle_mesh(str(path))
    return mesh


def merge_meshes(mesh_paths):
    vertices_all = []
    triangles_all = []
    v_offset = 0

    for p in mesh_paths:
        mesh = load_mesh_o3d(p)
        v = np.asarray(mesh.vertices)
        f = np.asarray(mesh.triangles)

        if len(v) == 0 or len(f) == 0:
            continue

        vertices_all.append(v)
        triangles_all.append(f + v_offset)
        v_offset += len(v)

    if len(vertices_all) == 0 or len(triangles_all) == 0:
        return None

    merged = o3d.geometry.TriangleMesh()
    merged.vertices = o3d.utility.Vector3dVector(np.vstack(vertices_all))
    merged.triangles = o3d.utility.Vector3iVector(np.vstack(triangles_all))
    merged.compute_triangle_normals()
    merged.compute_vertex_normals()
    return merged


def get_sample_id(sample_dir: Path):
    return sample_dir.name


def find_chair_meta_paths(partnet_root):
    partnet_root = Path(partnet_root)
    meta_paths = list(partnet_root.glob("*/meta.json"))

    chair_paths = []
    for p in meta_paths:
        try:
            meta = load_json(p)
            if meta.get("model_cat") == "Chair":
                chair_paths.append(p)
        except Exception:
            continue

    return chair_paths


def get_leg_nodes(sample_dir):
    result_path = Path(sample_dir) / "result_after_merging.json"
    tree = load_json(result_path)

    leg_nodes = []
    collect_nodes_by_name(tree, "leg", leg_nodes)

    leg_nodes = [node for node in leg_nodes if "objs" in node and len(node["objs"]) > 0]
    return leg_nodes


def build_one_sample(sample_dir, out_root, remove_strategy="first"):
    sample_dir = Path(sample_dir)
    out_root = Path(out_root)

    obj_dir = sample_dir / "objs"
    all_obj_paths = sorted(obj_dir.glob("*.obj"))
    if len(all_obj_paths) == 0:
        return False, f"{sample_dir.name}: no obj files"

    leg_nodes = get_leg_nodes(sample_dir)
    if len(leg_nodes) == 0:
        return False, f"{sample_dir.name}: no leg nodes"

    if remove_strategy == "random":
        target_leg = random.choice(leg_nodes)
    else:
        target_leg = leg_nodes[0]

    remove_objs = [name + ".obj" for name in target_leg["objs"]]

    M_gt = merge_meshes(all_obj_paths)
    if M_gt is None:
        return False, f"{sample_dir.name}: failed to build M_gt"

    kept_paths = [p for p in all_obj_paths if p.name not in remove_objs]
    M_d = merge_meshes(kept_paths)
    if M_d is None:
        return False, f"{sample_dir.name}: failed to build M_d"

    sample_id = get_sample_id(sample_dir)
    sample_out_dir = out_root / sample_id
    sample_out_dir.mkdir(parents=True, exist_ok=True)

    o3d.io.write_triangle_mesh(str(sample_out_dir / "M_gt.obj"), M_gt)
    o3d.io.write_triangle_mesh(str(sample_out_dir / "M_d.obj"), M_d)

    meta_info = {
        "sample_id": sample_id,
        "source_dir": str(sample_dir),
        "model_cat": "Chair",
        "removed_part_name": "leg",
        "removed_obj_files": remove_objs,
        "num_vertices_M_gt": int(np.asarray(M_gt.vertices).shape[0]),
        "num_faces_M_gt": int(np.asarray(M_gt.triangles).shape[0]),
        "num_vertices_M_d": int(np.asarray(M_d.vertices).shape[0]),
        "num_faces_M_d": int(np.asarray(M_d.triangles).shape[0]),
    }

    with open(sample_out_dir / "meta.json", "w", encoding="utf-8") as f:
        json.dump(meta_info, f, indent=2, ensure_ascii=False)

    return True, sample_id

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
from pathlib import Path
import random
partnetPath="E:/datasets/partnet/data_v0/data_v0"#replace your partnet path
partnet_root = Path(partnetPath)
dataset_root = Path("../dataset/")
dataset_root.mkdir(exist_ok=True)

random.seed(42)

In [3]:
chair_paths = find_chair_meta_paths(partnet_root)
print("num chair samples:", len(chair_paths))
print(chair_paths[:5])

num chair samples: 8176
[WindowsPath('E:/datasets/partnet/data_v0/data_v0/1095/meta.json'), WindowsPath('E:/datasets/partnet/data_v0/data_v0/1127/meta.json'), WindowsPath('E:/datasets/partnet/data_v0/data_v0/1277/meta.json'), WindowsPath('E:/datasets/partnet/data_v0/data_v0/1282/meta.json'), WindowsPath('E:/datasets/partnet/data_v0/data_v0/1284/meta.json')]


In [4]:
max_samples = 50
generated = 0
skipped = 0

chair_paths_shuffled = chair_paths.copy()
random.shuffle(chair_paths_shuffled)

for meta_path in chair_paths_shuffled:
    if generated >= max_samples:
        break

    sample_dir = meta_path.parent

    ok, msg = build_one_sample(
        sample_dir=sample_dir,
        out_root=dataset_root,
        remove_strategy="first"   # 可改成 "random"
    )

    if ok:
        generated += 1
        print(f"[OK] {generated:02d}/{max_samples} -> {msg}")
    else:
        skipped += 1
        print(f"[SKIP] {msg}")

print("done")
print("generated:", generated)
print("skipped:", skipped)
print("dataset root:", dataset_root.resolve())

[SKIP] 530: no leg nodes
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[OK] 01/50 -> 3278
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[OK] 02/50 -> 38518
[SKIP] 44463: no leg nodes
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[OK] 03/50 -> 37231
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[OK] 04/50 -> 42367
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[OK] 05/50 -> 39806
[SKIP] 40979: no leg nodes
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[OK] 06/50 -> 35618
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D W

In [5]:
sample_dirs = [p for p in dataset_root.iterdir() if p.is_dir()]
print("num generated dirs:", len(sample_dirs))
print(sample_dirs[:5])

num generated dirs: 100
[WindowsPath('../dataset/2101'), WindowsPath('../dataset/2168'), WindowsPath('../dataset/2210'), WindowsPath('../dataset/2269'), WindowsPath('../dataset/2398')]


In [6]:
sample_dir = sample_dirs[0]

M_gt = o3d.io.read_triangle_mesh(str(sample_dir / "M_gt.obj"))
M_d = o3d.io.read_triangle_mesh(str(sample_dir / "M_d.obj"))

M_gt.compute_triangle_normals()
M_gt.compute_vertex_normals()
M_d.compute_triangle_normals()
M_d.compute_vertex_normals()

o3d.visualization.draw_geometries([M_gt], window_name="M_gt", mesh_show_back_face=True)
o3d.visualization.draw_geometries([M_d], window_name="M_d", mesh_show_back_face=True)